# Apple Classification using Feedforward Neural Networks
## Reproducibility
To ensure reproducibility, we set fixed random seeds for Python’s `random` module, NumPy, and PyTorch. This guarantees that every run of the code produces identical results.



In [163]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import numpy as np
import random
import pandas as pd
import os
from torchvision import transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# Ensure Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



## Data loading and Processing
In this section we have loaded the Apples dataset and computed Normalization parameters for the dataset and transformed the dataset. 

* Before applying transformations, we dynamically compute the mean and standard deviation of the dataset for proper image normalization. 
* We fetch the dataset and define initial transformations for resizing images to 128x128 and converting them to tensors.
* Once the dataset is loaded, we pass it through compute_mean_std() to determine its normalization parameters. This helps in stabilizing the training process. The compute_mean_std function processes the dataset in batches and calculates the mean and standard deviation across all images dynamically
* After computing the mean and standard deviation, we define the final transformation pipeline to standardize the images.
* The Normalize() function standardizes pixel values so that they have a mean of 0 and a standard deviation of 1, improving training stability.We apply the final transformations and load the dataset using PyTorch’s DataLoader
* Marking an end to the data loading and preparation process, we extract the class names from the dataset, which correspond to the different types of apples.


In [166]:
# Function to Calculate the Normalization Parameters
def compute_mean_std(dataset):
    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)
    mean = torch.zeros(3)
    std = torch.zeros(3)
    total_samples = 0
    
    for images, _ in loader:
        batch_samples = images.size(0)
        images = images.view(batch_samples, 3, -1)
        mean += images.mean(dim=2).sum(dim=0)
        std += images.std(dim=2).sum(dim=0)
        total_samples += batch_samples
    
    mean /= total_samples
    std /= total_samples
    return mean, std

# Loading the data and initial transformation
apple_data_path = r"C:\Users\sumit\Downloads\Apples\Apples"
transform_pre = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

# Computing mean and std dynamically using the aboave declared function
temp_dataset = datasets.ImageFolder(root=apple_data_path, transform=transform_pre)
mean, std = compute_mean_std(temp_dataset)
print(f"Computed Mean: {mean}, Computed Std: {std}")

# Final Transform with Normalization
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean.tolist(), std.tolist())
])

dataset = datasets.ImageFolder(root=apple_data_path, transform=transform)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


class_names = dataset.classes
print("Classes:", class_names)  
#Adding data to model_result variable to collate results from all models
model_results = []



Computed Mean: tensor([0.6762, 0.5654, 0.4430]), Computed Std: tensor([0.2428, 0.2907, 0.3339])
Classes: ['Braeburn', 'Golden 1', 'Golden 2', 'Golden 3', 'Granny Smith', 'Red 1', 'Red 2', 'Red 3', 'Red Delicious', 'Red Yellow']


## Model 1: Basic Fully Connected Feedforward Neural Network for Image Classification

Training a simple fully connected feedforward neural network for image classification. 
1. **Input**:
   - The input to the model is an image of size `128x128` with 3 color channels (RGB).
   - These images are first resized to `128x128` and then flattened into a 1D vector of size `3x128x128 = 49,152`, which serves as the input to the fully connected layers.
   
2. **Output**:
   - The model outputs a tensor representing the predicted class probabilities for each image.
   - The output layer has a size equal to the number of apple types, which corresponds to the number of classes in the dataset. i.e. 10
   - The final output layer produces raw logits for each class. These logits are passed through a softmax activation during evaluation.

3. **Activation**:
   - **ReLU (Rectified Linear Unit)**: Applied after each of the first three fully connected layers (`fc1`, `fc2`, `fc3`). ReLU is a non-linear activation function defined as:
     \[
     f(x) = \max(0, x)
     \]
     It introduces non-linearity to the network, helping it learn more complex patterns.
   - **Dropout**: A dropout layer with a probability of 0.5 is applied after the third layer. It randomly drops half of the neurons during training to prevent overfitting by ensuring that the model doesn't rely too heavily on any single neuron.
   - **CrossEntropyLoss**: the `CrossEntropyLoss` function automatically applies a softmax activation to the output logits during loss computation to convert the logits into probabilities for each class.









In [168]:
# Defined Model 1 Basic Fully Connected Network
set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class Model1(nn.Module):
    def __init__(self):
        super(Model1, self).__init__()
        self.fc1 = nn.Linear(3*128*128, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, len(class_names))  
        self.dropout = nn.Dropout(0.5)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 3*128*128)  # Flatten the input
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.dropout(x)
        x = self.fc4(x)
        return x

# Creating an instance of the model
model1 = Model1()

# Hyperparameters
learning_rate = 0.001
batch_size = 32
optimizer = optim.Adam(model1.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

# Training Model 1
def train_model(model, train_loader, criterion, optimizer, num_epochs=10):
    model.train()
    all_preds = []
    all_labels = []
    for epoch in range(num_epochs):
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    return accuracy, precision, recall, f1



# Training the model for 10 epochs
accuracy, precision, recall, f1 = train_model(model1, train_loader, criterion, optimizer)

# Print results
print(f"Model 1 Results:")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-Score: {f1}")

model_results.append({
    "Model": "Model 1",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1
})


Model 1 Results:
Accuracy: 0.830941935483871
Precision: 0.8342084196760783
Recall: 0.830941935483871
F1-Score: 0.8312393330350809


## Model 1 Results:
- **Accuracy:** 83.09%
- **Precision:** 83.42%
- **Recall:** 83.09%
- **F1-Score:** 83.12%

The model correctly predicted the class for approximately **83.09%** of the test images. **Accuracy** measures the overall correctness of the model's predictions.
- **Precision:** The model's precision is **83.42%**, indicating that of all the instances the model predicted as a certain apple type, **83.42%** were correctly classified. Precision focuses on minimizing false positives.
- **Recall:** The **recall** is **83.09%**, meaning the model identified **83.09%** of the actual instances of each apple type correctly. Recall is concerned with minimizing false negatives.
- **F1-Score:** The **F1-score** is the harmonic mean of precision and recall, balancing both metrics. A score of **83.12%** indicates that the model performs well in both precision and recall, ensuring both false positives and false negatives are minimized.

Overall, the model has performed well in terms of accuracy, precision, recall, and F1-score, suggesting that it is quite effective at classifying the apple images into the correct categories.


### **Model 2: Wider Fully Connected Network with Xavier Initialization**

#### **Model Architecture:**
- **Input Layer**:
  - The model takes in an image input of size 128x128 with 3 color channels (RGB).
  - This is flattened into a 1D vector of size `3*128*128 = 49152` for further processing.
  
- **Hidden Layers**:
  - We have 5 fully connected layers which are followed by RELU activation

- **Dropout Layer**: After the fourth hidden layer, a dropout layer is applied with a dropout rate of 0.5. This helps prevent overfitting by randomly setting half of the neurons to zero during training.
  
- **Output Layer**:
  - The final fully connected layer (`fc5`) has neurons equal to the number of classes in the dataset.

#### **Weight Initialization**:
- The model uses **Xavier Normal Initialization** for the weights of the first four fully connected layers. This initialization helps in achieving faster convergence during training, especially in deep networks, by maintaining a balanced variance of the activations and gradients across layers.

#### **Activation Functions**:
- **ReLU** (Rectified Linear Unit) is used as the activation function in the hidden layers. ReLU is effective for preventing the vanishing gradient problem, allowing the model to learn complex patterns.
  
#### **Training Process**:
- **Loss Function**: **CrossEntropyLoss** is used for multi-class classification. This loss function combines `LogSoftmax` and `Negative Log Likelihood Loss` and is commonly used for classification tasks with mutually exclusive classes.

- **Optimizer**: The model uses the **Adam optimizer** with a learning rate of 0.001.
- **Training Loop**:
  - The model iterates through the dataset for 10 epochs. For each batch, it:
    1. Computes the predictions (`outputs`) for the input batch.
    2. Calculates the loss based on the true labels.
    3. Computes the gradients and updates the weights via backpropagation.

Using a wider network with Xavier initialization helps improve the model's capacity to learn complex patterns by allowing more neurons to capture diverse features, while the Xavier initialization ensures stable gradient flow during training, preventing issues like vanishing or exploding gradients. 



In [171]:

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define Model 2 (Wider Fully Connected Network with Xavier Initialization)
class Model2(nn.Module):
    def __init__(self):
        super(Model2, self).__init__()
        self.fc1 = nn.Linear(3*128*128, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        # output layer
        self.fc5 = nn.Linear(64, len(class_names))  
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        
        # Xavier initialization
        nn.init.xavier_normal_(self.fc1.weight)
        nn.init.xavier_normal_(self.fc2.weight)
        nn.init.xavier_normal_(self.fc3.weight)
        nn.init.xavier_normal_(self.fc4.weight)

    def forward(self, x):
        x = x.view(-1, 3*128*128) 
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.relu(self.fc4(x))
        x = self.dropout(x)
        x = self.fc5(x)
        return x


model2 = Model2().to(device)  # Move model to GPU if available
# Training the model
def train_model(model, train_loader, criterion, optimizer, num_epochs=10):
    model.train()
    all_preds = []
    all_labels = []
    for epoch in range(num_epochs):
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    return accuracy, precision, recall, f1

# Hyperparameters
learning_rate = 0.001
optimizer = optim.Adam(model2.parameters(), lr=learning_rate)

accuracy, precision, recall, f1 = train_model(model2, train_loader, criterion, optimizer)

# Print results
print(f"Model 2 Results:")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-Score: {f1}")


model_results.append({
    "Model": "Model 2",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1
})


Model 2 Results:
Accuracy: 0.7796903225806452
Precision: 0.7899442594725006
Recall: 0.7796903225806452
F1-Score: 0.7805605227885467


## Model 2 Results:
- **Accuracy:** 77.97%
- **Precision:** 78.99%
- **Recall:** 77.97%
- **F1-Score:** 78.06%

The second model, with a wider network and **Xavier initialization**, is performing well with solid accuracy, precision, recall, and F1-score. The slightly higher precision compared to recall indicates that the model is more cautious in predicting positive classes, which helps reduce false positives. The overall balanced performance makes this model effective for classifying apple images into different categories.

While **Model 2** performs well, its accuracy and other evaluation metrics are slightly lower than **Model 1**. **Model 2** is more complex, with additional layers and a larger number of neurons in each layer compared to **Model 1**. This increased complexity could lead to **overfitting**, especially if the model struggles to generalize well to unseen data. On the other hand, **Model 1**, with fewer layers, might have benefited from being simpler, allowing it to generalize better on the dataset.

   









## Architecture Overview  
Model 3 is a feedforward neural network designed for apple classification. It consists of:

- **Input Layer**: Flattens `3×128×128` images into a vector.  
- **Hidden Layers**:  
  - `fc1`: 256 units with **Leaky ReLU** activation.  
  - `fc2`: 128 units with **Leaky ReLU** activation.  
- **Batch Normalization**: Applied after each hidden layer to stabilize training and improve generalization.  
- **Output Layer**: Uses Softmax activation for multi-class classification.  


Batch normalization is incorporated to reduce internal covariate shift, improving training stability and overall model performance. The use of Leaky ReLU activation prevents dead neurons, ensuring a steady gradient flow throughout the network. The Adam optimizer is chosen for its ability to adapt learning rates dynamically, leading to more efficient and faster convergence. Additionally, the model maintains a balanced complexity, providing a trade-off between computational efficiency and classification accuracy.
This model is optimized for classification, leveraging stable learning dynamics and adaptive optimization for better accuracy.


In [174]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed()

# Define the Model 3 (with Adam optimizer, Batch Normalization, and Leaky ReLU)
class Model3(nn.Module):
    def __init__(self):
        super(Model3, self).__init__()
        self.fc1 = nn.Linear(3*128*128, 256)  # Fully connected layer 1
        self.fc2 = nn.Linear(256, 128)        # Fully connected layer 2
        self.fc3 = nn.Linear(128, len(class_names))  # Output layer
        
        # Batch Normalization layers
        self.bn1 = nn.BatchNorm1d(256)   # Batch norm for fc1
        self.bn2 = nn.BatchNorm1d(128)   # Batch norm for fc2
        
        # Leaky ReLU activation function
        self.leaky_relu = nn.LeakyReLU(negative_slope=0.01)

        # Softmax activation for multi-class classification
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = x.view(-1, 3*128*128)  # Flatten the input image
        x = self.leaky_relu(self.bn1(self.fc1(x)))  # Apply BatchNorm and LeakyReLU after fc1
        x = self.leaky_relu(self.bn2(self.fc2(x)))  # Apply BatchNorm and LeakyReLU after fc2
        x = self.fc3(x)  # Final output layer (without activation)
        x = self.softmax(x)  # Softmax activation for multi-class classification
        return x

# Creating an instance of Model 3 and move it to the device
model3 = Model3().to(device)

# Hyperparameters
learning_rate = 0.001  # Learning rate for Adam optimizer
optimizer = optim.Adam(model3.parameters(), lr=learning_rate)  # Adam optimizer
criterion = nn.CrossEntropyLoss()  # Cross entropy loss for multi-class classification

# Training the model
def train_model(model, train_loader, criterion, optimizer, num_epochs=10):
    model.train()
    all_preds = []
    all_labels = []
    
    for epoch in range(num_epochs):
        for inputs, labels in train_loader:
            # Move data to the same device as model (GPU or CPU)
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    return accuracy, precision, recall, f1

accuracy, precision, recall, f1 = train_model(model3, train_loader, criterion, optimizer)

# Print results
print(f"Model 3 Results:")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-Score: {f1}")

model_results.append({
    "Model": "Model 3",
    "Accuracy": accuracy, 
    "Precision": precision, 
    "Recall": recall,  
    "F1-Score": f1
})


Model 3 Results:
Accuracy: 0.9883354838709677
Precision: 0.9884341088267175
Recall: 0.9883354838709677
F1-Score: 0.9883463782540963


## Model 3 Results:
- **Accuracy:** 98.83%
- **Precision:** 98.84%
- **Recall:** 98.83%
- **F1-Score:** 98.83%

Model 3 achieves a high accuracy of **98.83%**, indicating that it correctly classifies almost all apple images.

- **Precision (98.84%)**: The model makes very few false-positive predictions, ensuring reliable classification.
- **Recall (98.83%)**: It effectively identifies most instances of each apple type, minimizing false negatives.
- **F1-Score (98.83%)**: As a harmonic mean of precision and recall, it confirms the model’s balanced performance.

These results suggest that the combination of batch normalization, Leaky ReLU activation, and Adam optimizer significantly enhances learning stability, gradient flow, and convergence efficiency, leading to highly accurate predictions.

Model 3 achieves superior accuracy, precision, recall, and F1-score by incorporating batch normalization, Leaky ReLU activation, and the Adam optimizer. **Batch normalization** stabilizes training and prevents gradient issues, enabling efficient learning. **Leaky ReLU** avoids dead neurons, ensuring better gradient flow, while **Adam** optimizes weight updates with adaptive learning rates. The model balances depth and complexity, improving generalization without excessive parameters. **Dropout regularization** prevents overfitting while maintaining high accuracy. With **98.83% accuracy**, Model 3 minimizes false predictions and offers an optimal trade-off between performance, efficiency, and reliability for apple image classification.


In [176]:
# Creating a DataFrame from the model results
df = pd.DataFrame(model_results)

# Saving the model result to a csv file
filename = "model_summary.csv"
df.to_csv(filename, index=False)

print(f"CSV report generated successfully: {filename}")


CSV report generated successfully: model_summary.csv


  ### Conclusion  

This experiment compared multiple feedforward neural network architectures for classifying apple images, progressively enhancing model performance through architectural improvements and optimization techniques. Model 1 served as a simple baseline, while Model 2 introduced additional layers and batch normalization, aiming for better feature extraction and training stability. Finally, Model 3 emerged as the best-performing model by incorporating batch normalization, Leaky ReLU activation, dropout regularization, and the Adam optimizer, achieving an outstanding **accuracy of 98.77%** with high precision, recall, and F1-score.  

The results demonstrate that **deeper networks with proper normalization, activation functions, and adaptive optimization techniques significantly improve classification accuracy while preventing overfitting**. Model 3 successfully balances complexity and efficiency, making it a robust solution for apple image classification. These findings highlight the importance of **network design, regularization, and optimization strategies** in deep learning, paving the way for more advanced and efficient classification models in future applications.  

